# Current metadata probe: frozen candidate IDs
Observation: 2026-09-12 08:32 UTC. This notebook verifies the committed probe receipts without new API calls. It does not rerun the model, change qualification, or infer the historical cause of missing metadata. Raw responses remain local; set `MSOS_METADATA_PROBE_OBSERVATIONS` to verify their hashes.


In [1]:
import hashlib, json, os
from pathlib import Path
repo = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "docs/handoff-blueprint.md").exists())
report = json.loads((repo / "docs/benchmarks/2026-09-12-metadata-probe.json").read_text())
audit_path = repo / "docs/benchmarks" / report["source_audit"]["name"]
audit_bytes = audit_path.read_bytes()
# Git checkouts may convert Windows CRLF to LF; accept only those equivalent bytes.
def newline_hashes(data):
    lf = data.replace(b"\r\n", b"\n")
    return {hashlib.sha256(value).hexdigest() for value in (data, lf, lf.replace(b"\n", b"\r\n"))}
assert report["source_audit"]["sha256"] in newline_hashes(audit_bytes)
audit = json.loads(audit_bytes)
candidates = [row for row in audit["wallets"] if row["metadata_only_candidate"]]
expected_ids = {cid for row in candidates for cid in row["missing_condition_ids"]}
assert set(report["condition_ids"]) == expected_ids
for name, digest in report["source_code_sha256"].items():
    code_path = repo / "services/polymarket-ingestor/src/marketsignalos_polymarket" / name
    assert digest in newline_hashes(code_path.read_bytes()), name
print("Audit and producing code verified;", len(expected_ids), "frozen candidate IDs.")

Audit and producing code verified; 41 frozen candidate IDs.


In [2]:
states = report["condition_observations"]
assert len({(row["condition_id"], row["closed"]) for row in states}) == len(states)
returned = {row["condition_id"] for row in states if row["outcome"] == "stored"}
absent = {cid for cid in expected_ids if sum(row["condition_id"] == cid and row["outcome"] == "not_returned" for row in states) == 2}
summary = {"conditions": len(expected_ids), "returned_now": len(returned), "not_returned_either_filter": len(absent), "inconclusive": len(expected_ids - returned - absent)}
assert summary == report["summary"]
assert len(report["attempts"]) == report["run"]["requests"] == 4
assert all(row["http_status"] == 200 and row["outcome"] == "recorded" for row in report["attempts"])
assert len(states) == report["run"]["attempted_lookups"] == 82
assert {cid for row in report["attempts"] for cid in json.loads(row["condition_ids"])} == expected_ids
for row in candidates:
    ids = set(row["missing_condition_ids"])
    print(row["wallet"], {"missing": len(ids), "returned_now": len(ids & returned), "not_returned": len(ids & absent)})
print(summary)
print("Current availability only. Historical absence cause and score impact are unestablished.")

0x521070c99db06e54af5e8e4a91d6858decdfbd53 {'missing': 36, 'returned_now': 32, 'not_returned': 4}
0x5a218c7ad04135830a45c41aaed7294df7809318 {'missing': 5, 'returned_now': 4, 'not_returned': 1}
{'conditions': 41, 'returned_now': 36, 'not_returned_either_filter': 5, 'inconclusive': 0}
Current availability only. Historical absence cause and score impact are unestablished.


In [3]:
observations_path = os.environ.get("MSOS_METADATA_PROBE_OBSERVATIONS")
if observations_path:
    raw = Path(observations_path).read_bytes()
    assert hashlib.sha256(raw).hexdigest() == report["observations_sha256"]
    observations = [json.loads(line)["market"] for line in raw.splitlines()]
    assert len(observations) == report["run"]["rows_written"]
    for attempt in report["attempts"]:
        batch = set(json.loads(attempt["condition_ids"]))
        response = [row for row in observations if row["conditionId"] in batch and row["closed"] == bool(attempt["closed"])]
        canonical = json.dumps(response, sort_keys=True, allow_nan=False, separators=(",", ":")).encode()
        assert hashlib.sha256(canonical).hexdigest() == attempt["response_sha256"]
        assert {row["conditionId"] for row in response} == set(json.loads(attempt["returned_ids"]))
    print("Raw observation file and all four response hashes verified; no network calls.")
else:
    print("Raw response verification skipped: set MSOS_METADATA_PROBE_OBSERVATIONS to the local file.")

Raw observation file and all four response hashes verified; no network calls.
